In [4]:
import os
import re
import json
import time
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Tuple
from urllib.parse import urljoin, urlparse

In [5]:
import requests
from bs4 import BeautifulSoup

In [6]:
!pip install -U langchain langchain-core langchain-text-splitters langchain-chroma langchain-openai chromadb beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 23.8 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: chromadb
    Found existing installation: chromadb 1.5.6
    Uninstalling chromadb-1.5.6:
      Successfully uninstalled chromadb-1.5.6


In [7]:
!pip install -U sentence-transformers langchain-huggingface

In [8]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

/Users/annie/Desktop/Code/RSM8430/llm-assignment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Configuration: This section defines the global settings for crawling, chunking, retrieval, vector storage, and model access.

In [9]:
BASE_INDEX_URL = "https://travel.gc.ca/travelling/advisories"
USER_AGENT = "RSM8430H-ProjectBot/1.0 (educational use)"
REQUEST_TIMEOUT = 20
REQUEST_DELAY_SEC = 0.5
MAX_PAGES = None

PERSIST_DIRECTORY = "./chroma_db_canada_travel"
COLLECTION_NAME = "canada_travel_advisories"

CHUNK_SIZE = 700
CHUNK_OVERLAP = 120

DEFAULT_TOP_K = 4
DEFAULT_FETCH_K = 10
DEFAULT_MMR_LAMBDA = 0.5

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
CHAT_MODEL = os.getenv("CHAT_MODEL", "qwen3-30b-a3b-fp8")
LOCAL_EMBEDDING_MODEL = os.getenv(
    "LOCAL_EMBEDDING_MODEL",
    "sentence-transformers/all-MiniLM-L6-v2"
)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")

# 2. Data model ("AdvisoryPage"): This dataclass stores each advisory page in a clean and structured format.

In [10]:
@dataclass
class AdvisoryPage:
    url: str
    title: str
    destination: str
    risk_level: str
    last_updated: str
    sections: List[Tuple[str, str]]

# 3. Networking helpers: These helper functions manage HTTP requests and safely download raw HTML pages.

In [11]:
def make_session() -> requests.Session:
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})
    return session

def fetch_html(session: requests.Session, url: str) -> str:
    response = session.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    time.sleep(REQUEST_DELAY_SEC)
    return response.text

# 4. Crawling helpers:This section extracts and cleans the destination page URLs that will be used to build the knowledge base.

In [12]:
def normalize_url(url: str) -> str:
    parsed = urlparse(url)
    normalized = parsed._replace(query="", fragment="")
    return normalized.geturl().rstrip("/")


def looks_like_destination_page(url: str) -> bool:
    url = normalize_url(url)
    return "/destinations/" in url and url.startswith("https://travel.gc.ca")


def extract_destination_links(index_html: str, base_url: str) -> List[str]:
    soup = BeautifulSoup(index_html, "html.parser")
    links: List[str] = []
    seen = set()

    for a_tag in soup.find_all("a", href=True):
        href = urljoin(base_url, a_tag["href"])
        href = normalize_url(href)
        if looks_like_destination_page(href) and href not in seen:
            seen.add(href)
            links.append(href)

    return links

# 5. Parsing helpers:


5.1. This function normalizes raw text by removing extra whitespace.

In [13]:
def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()

5.2 This function tries to locate the main content area of each webpage and ignore page noise.

In [14]:
def find_main_content(soup: BeautifulSoup) -> BeautifulSoup:
    selectors = [
        "main",
        "article",
        "#wb-cont",
        ".mwsgeneric-base-html",
        ".gc-byline",
        ".content",
    ]
    for selector in selectors:
        node = soup.select_one(selector)
        if node is not None:
            return node
    return soup

5.3 This function extracts the page title to identify the advisory page clearly.

In [15]:
def extract_title(soup: BeautifulSoup) -> str:
    h1 = soup.find("h1")
    if h1:
        return clean_text(h1.get_text(" ", strip=True))
    if soup.title:
        return clean_text(soup.title.get_text(" ", strip=True))
    return "Untitled advisory"

5.4 This function identifies the official travel risk level from the page text.

In [16]:
def extract_risk_level(text: str) -> str:
    patterns = [
        r"(Exercise normal security precautions)",
        r"(Exercise a high degree of caution)",
        r"(Avoid non-essential travel)",
        r"(Avoid all travel)",
        r"(Take normal security precautions)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return clean_text(match.group(1))
    return "Unknown"

5.5 This function extracts the page’s last updated date for freshness tracking.

In [17]:
def extract_last_updated(text: str) -> str:
    patterns = [
        r"Date modified:\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})",
        r"Last updated:\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})",
        r"Updated:\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return clean_text(match.group(1))
    return "Unknown"

5.6 This function groups webpage content into meaningful sections based on headings, which improves semantic chunk quality.

In [18]:
def extract_sections(main_node: BeautifulSoup) -> List[Tuple[str, str]]:

    sections: List[Tuple[str, str]] = []

    current_heading = "Overview"
    current_parts: List[str] = []

    for element in main_node.find_all(["h2", "h3", "p", "li"]):
        if element.name in {"h2", "h3"}:
            section_text = clean_text(" ".join(current_parts))
            if section_text:
                sections.append((current_heading, section_text))
            current_heading = clean_text(element.get_text(" ", strip=True)) or "Unnamed section"
            current_parts = []
        else:
            text = clean_text(element.get_text(" ", strip=True))
            if text:
                current_parts.append(text)

    final_text = clean_text(" ".join(current_parts))
    if final_text:
        sections.append((current_heading, final_text))

    sections = [(h, t) for h, t in sections if len(t) >= 80]
    return sections

5.7 This function converts a raw advisory HTML page into a structured [AdvisoryPage] object.

In [19]:
def parse_advisory_page(html: str, url: str) -> AdvisoryPage:
    soup = BeautifulSoup(html, "html.parser")
    main_node = find_main_content(soup)
    page_text = clean_text(main_node.get_text(" ", strip=True))

    title = extract_title(soup)
    destination = title.replace("Travel advice and advisories for ", "").strip()
    risk_level = extract_risk_level(page_text)
    last_updated = extract_last_updated(page_text)
    sections = extract_sections(main_node)

    return AdvisoryPage(
        url=url,
        title=title,
        destination=destination,
        risk_level=risk_level,
        last_updated=last_updated,
        sections=sections,
    )

# 6. Build structured documents:

6.1 This function crawls multiple advisory pages and parses them into structured page objects.

In [20]:
def crawl_and_parse(
    base_index_url: str = BASE_INDEX_URL,
    max_pages: Optional[int] = MAX_PAGES
) -> List[AdvisoryPage]:
    session = make_session()
    index_html = fetch_html(session, base_index_url)
    candidate_links = extract_destination_links(index_html, base_index_url)

    if max_pages is not None:
        candidate_links = candidate_links[:max_pages]

    pages: List[AdvisoryPage] = []
    for url in candidate_links:
        try:
            html = fetch_html(session, url)
            page = parse_advisory_page(html, url)
            if page.sections:
                pages.append(page)
                print(f"[OK] Parsed: {page.destination} | sections={len(page.sections)}")
            else:
                print(f"[SKIP] No meaningful sections: {url}")
        except Exception as exc:
            print(f"[ERROR] Failed to parse {url}: {exc}")

    return pages

6.2 This function turns parsed advisory pages into LangChain documents with content and metadata.

In [21]:
def advisory_pages_to_documents(pages: List[AdvisoryPage]) -> List[Document]:
    documents: List[Document] = []

    for page in pages:
        for heading, section_text in page.sections:
            metadata = {
                "source": page.url,
                "url": page.url,
                "title": page.title,
                "destination": page.destination,
                "risk_level": page.risk_level,
                "last_updated": page.last_updated,
                "section_title": heading,
            }

            content = (
                f"Destination: {page.destination}\n"
                f"Risk level: {page.risk_level}\n"
                f"Section: {heading}\n\n"
                f"{section_text}"
            )
            documents.append(Document(page_content=content, metadata=metadata))

    return documents

6.3 This function splits section-level documents into smaller retrieval-friendly chunks.

In [22]:
def split_documents(documents: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_documents(documents)

6.4 This function saves the parsed advisory data as JSON for inspection and debugging.

In [23]:
def save_pages_json(pages: List[AdvisoryPage], path: str = "parsed_pages.json") -> None:
    payload = [
        {
            "url": page.url,
            "title": page.title,
            "destination": page.destination,
            "risk_level": page.risk_level,
            "last_updated": page.last_updated,
            "sections": [{"heading": h, "text": t} for h, t in page.sections],
        }
        for page in pages
    ]
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

# 7. Vector store / embeddings：

7.1 This function initializes the embedding model used to convert text chunks into vectors.

In [24]:
def get_embeddings() -> Any:
    if OPENAI_API_KEY:
        kwargs: Dict[str, Any] = {
            "model": EMBEDDING_MODEL,
            "api_key": OPENAI_API_KEY,
        }
        if OPENAI_BASE_URL:
            kwargs["base_url"] = OPENAI_BASE_URL
        return OpenAIEmbeddings(**kwargs)

    print(f"[INFO] OPENAI_API_KEY not found. Falling back to local embeddings: {LOCAL_EMBEDDING_MODEL}")
    return HuggingFaceEmbeddings(model_name=LOCAL_EMBEDDING_MODEL)

7.2 This function builds the Chroma vector store from chunked documents and embeddings.

In [25]:
def build_vectorstore(chunks: List[Document], persist_directory: str = PERSIST_DIRECTORY) -> Chroma:
    embeddings = get_embeddings()

    if os.path.exists(persist_directory):
        import shutil
        shutil.rmtree(persist_directory)

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=COLLECTION_NAME,
    )
    return vectorstore

7.3 This function loads an existing vector store so the pipeline can reuse the indexed knowledge base.

In [26]:
def load_vectorstore(persist_directory: str = PERSIST_DIRECTORY) -> Chroma:
    embeddings = get_embeddings()
    return Chroma(
        collection_name=COLLECTION_NAME,
        persist_directory=persist_directory,
        embedding_function=embeddings,
    )

# 8. Retrieval：This function retrieves the most relevant document chunks for a user query from the vector store.

In [27]:
def retrieve_docs(
    query: str,
    vectorstore: Chroma,
    k: int = DEFAULT_TOP_K,
    use_mmr: bool = True,
    destination_filter: Optional[str] = None,
) -> List[Document]:
    search_kwargs: Dict[str, Any] = {"k": k}

    if use_mmr:
        search_kwargs["fetch_k"] = max(DEFAULT_FETCH_K, k * 2)
        search_kwargs["lambda_mult"] = DEFAULT_MMR_LAMBDA

    if destination_filter:
        search_kwargs["filter"] = {"destination": destination_filter}

    retriever = vectorstore.as_retriever(
        search_type="mmr" if use_mmr else "similarity",
        search_kwargs=search_kwargs,
    )
    return retriever.invoke(query)

# 9. QA prompt：

9.1 This prompt instructs the model to answer only from retrieved evidence and avoid unsupported claims.

In [28]:
QA_SYSTEM_PROMPT = """
You are a travel advisory assistant.
Answer ONLY using the retrieved context.
If the answer is not supported by the context, say you do not have enough information.
Always be careful with safety-critical advice.
When possible, cite the supporting destination and section title in brackets like:
[France - Entry requirements]
Keep answers concise, factual, and grounded.
""".strip()

9.2 This function initializes the chat model used for grounded answer generation.

In [29]:
def get_chat_model(temperature: float = 0.1) -> ChatOpenAI:
    if not OPENAI_API_KEY:
        raise ValueError(
            "OPENAI_API_KEY is not set. Retrieval can run locally, but answer generation still needs a hosted chat model."
        )

    kwargs: Dict[str, Any] = {
        "model": CHAT_MODEL,
        "api_key": OPENAI_API_KEY,
        "temperature": temperature,
    }
    if OPENAI_BASE_URL:
        kwargs["base_url"] = OPENAI_BASE_URL

    return ChatOpenAI(**kwargs)

9.3 This function formats retrieved documents into a structured context block for the language model.

In [30]:
def format_context(docs: List[Document]) -> str:
    parts = []
    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata
        parts.append(
            f"[Document {i}]\n"
            f"Destination: {meta.get('destination', 'Unknown')}\n"
            f"Section: {meta.get('section_title', 'Unknown')}\n"
            f"Risk level: {meta.get('risk_level', 'Unknown')}\n"
            f"URL: {meta.get('url', '')}\n"
            f"Content: {doc.page_content}\n"
        )
    return "\n\n".join(parts)

9.4 This function runs the end-to-end RAG question answering pipeline and returns both the answer and its sources.

In [31]:
def answer_question(
    query: str,
    vectorstore: Chroma,
    k: int = DEFAULT_TOP_K,
    use_mmr: bool = True,
    destination_filter: Optional[str] = None,
) -> Dict[str, Any]:
    docs = retrieve_docs(
        query=query,
        vectorstore=vectorstore,
        k=k,
        use_mmr=use_mmr,
        destination_filter=destination_filter,
    )

    sources = [
        {
            "destination": d.metadata.get("destination"),
            "section_title": d.metadata.get("section_title"),
            "url": d.metadata.get("url"),
            "risk_level": d.metadata.get("risk_level"),
        }
        for d in docs
    ]

    if not OPENAI_API_KEY:
        fallback_lines = []
        for d in docs[: min(3, len(docs))]:
            section = d.metadata.get("section_title", "Unknown section")
            destination = d.metadata.get("destination", "Unknown destination")
            snippet = d.page_content[:300].replace("\n", " ")
            fallback_lines.append(f"[{destination} - {section}] {snippet}...")

        fallback_answer = (
            "API key not configured, so full LLM generation is disabled. "
            "Below are the most relevant retrieved passages that support the answer:\n\n"
            + "\n".join(fallback_lines)
        )

        return {
            "query": query,
            "answer": fallback_answer,
            "sources": sources,
            "retrieved_docs": docs,
        }

    llm = get_chat_model()
    context = format_context(docs)
    user_prompt = (
        f"User question:\n{query}\n\n"
        f"Retrieved context:\n{context}\n\n"
        "Write a grounded answer based only on the context above."
    )

    response = llm.invoke([
        {"role": "system", "content": QA_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ])

    return {
        "query": query,
        "answer": response.content,
        "sources": sources,
        "retrieved_docs": docs,
    }

# 10. Evaluation helpers：This function evaluates retrieval quality by measuring whether expected evidence appears in the retrieved results.

In [32]:
def run_retrieval_experiment(
    test_cases: List[Dict[str, Any]],
    vectorstore: Chroma,
    k_values: Iterable[int] = (2, 4, 6),
    use_mmr_options: Iterable[bool] = (False, True),
) -> List[Dict[str, Any]]:
    """
    Simple retrieval-side experiment.
    Each test case can contain:
    {
        "query": "...",
        "expected_destination": "France",
        "expected_section": "Entry and exit requirements"
    }

    This does not score the final answer text. It checks whether the
    expected destination/section appears in the retrieved set.
    """
    results: List[Dict[str, Any]] = []

    for k in k_values:
        for use_mmr in use_mmr_options:
            hit_count = 0
            total = 0

            for case in test_cases:
                total += 1
                docs = retrieve_docs(case["query"], vectorstore, k=k, use_mmr=use_mmr)

                expected_destination = case.get("expected_destination")
                expected_section = case.get("expected_section")

                hit = False
                for doc in docs:
                    meta = doc.metadata
                    destination_ok = (
                        expected_destination is None
                        or meta.get("destination") == expected_destination
                    )
                    section_ok = (
                        expected_section is None
                        or meta.get("section_title") == expected_section
                    )
                    if destination_ok and section_ok:
                        hit = True
                        break

                if hit:
                    hit_count += 1

            results.append(
                {
                    "k": k,
                    "use_mmr": use_mmr,
                    "retrieval_hit_rate": round(hit_count / total, 3) if total else 0.0,
                    "n_cases": total,
                }
            )

    return results

# 11. Main workflow： This function builds the full knowledge base pipeline from crawling to vector indexing.

In [33]:
def build_rag_pipeline() -> None:
    pages = crawl_and_parse()
    if len(pages) < 50:
        raise ValueError(
            f"Only parsed {len(pages)} pages. You need at least 50 documents/pages for the project."
        )

    save_pages_json(pages)
    docs = advisory_pages_to_documents(pages)
    chunks = split_documents(docs)
    print(f"Parsed pages: {len(pages)}")
    print(f"Base documents: {len(docs)}")
    print(f"Chunks: {len(chunks)}")

    build_vectorstore(chunks)
    print("Vector store built successfully.")

# 12. Example usage：This section shows how to build the knowledge base once and then use it for question answering.

In [34]:
if __name__ == "__main__":
    vectorstore = load_vectorstore()
    questions = [
        "What is the travel risk level for France?",
        "Do I need to prepare for health precautions before traveling to Brazil?",
        "What should I know about entry requirements for Japan?",
    ]

    for question in questions:
        result = answer_question(question, vectorstore, k=4, use_mmr=True)
        print("\n" + "=" * 80)
        print("QUESTION:", result["query"])
        print("ANSWER:", result["answer"])
        print("SOURCES:")
        for source in result["sources"]:
            print(source)

[INFO] OPENAI_API_KEY not found. Falling back to local embeddings: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9166.03it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



QUESTION: What is the travel risk level for France?
ANSWER: API key not configured, so full LLM generation is disabled. Below are the most relevant retrieved passages that support the answer:

[France travel advice - Disclaimer] Destination: France travel advice Risk level: Exercise a high degree of caution Section: Disclaimer...
[France travel advice - Crime] Destination: France travel advice Risk level: Exercise a high degree of caution Section: Crime...
[France travel advice - Avoid non-essential travel] Destination: France travel advice Risk level: Exercise a high degree of caution Section: Avoid non-essential travel  Your safety and security could be at risk. You should think about your need to travel to this country, territory or region based on family or business requirements, knowledge of or fa...
SOURCES:
{'destination': 'France travel advice', 'section_title': 'Disclaimer', 'url': 'https://travel.gc.ca/destinations/france', 'risk_level': 'Exercise a high degree of caution'}


In [36]:
from pathlib import Path

# VS Code / local Jupyter version (active)
pages = crawl_and_parse()
output_path = Path("parsed_pages.json")
save_pages_json(pages, path=str(output_path))

print(f"Saved {len(pages)} pages to: {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size:,} bytes")

# Google Colab version (commented)
# Uncomment this block when running in Colab.
# import json
# from google.colab import files
#
# pages = crawl_and_parse()
# payload = [
#     {
#         "url": page.url,
#         "title": page.title,
#         "destination": page.destination,
#         "risk_level": page.risk_level,
#         "last_updated": page.last_updated,
#         "sections": [{"heading": h, "text": t} for h, t in page.sections],
#     }
#     for page in pages
# ]
#
# with open("parsed_pages.json", "w", encoding="utf-8") as f:
#     json.dump(payload, f, ensure_ascii=False, indent=2)
#
# files.download("parsed_pages.json")

[OK] Parsed: Afghanistan travel advice | sections=48
[OK] Parsed: Albania travel advice | sections=51
[OK] Parsed: Algeria travel advice | sections=62
[OK] Parsed: American Samoa travel advice | sections=41
[OK] Parsed: Andorra travel advice | sections=39
[OK] Parsed: Angola travel advice | sections=47
[OK] Parsed: Anguilla travel advice | sections=47
[OK] Parsed: Antarctica travel advice | sections=26
[OK] Parsed: Antigua and Barbuda travel advice | sections=45
[OK] Parsed: Argentina travel advice | sections=59
[OK] Parsed: Armenia travel advice | sections=50
[OK] Parsed: Aruba travel advice | sections=45
[OK] Parsed: Australia travel advice | sections=43
[OK] Parsed: Austria travel advice | sections=46
[OK] Parsed: Azerbaijan travel advice | sections=52
[OK] Parsed: Azores travel advice | sections=48
[OK] Parsed: Bahamas travel advice | sections=50
[OK] Parsed: Bahrain travel advice | sections=51
[OK] Parsed: Bangladesh travel advice | sections=61
[OK] Parsed: Barbados travel advice 